# Lab 61: Grading multimodal RAG by stage

A multimodal answer can fail at retrieval, OCR-reading, or grounding, and one accuracy number can't tell them apart. Grade each axis and attribute every end-to-end error to its first failing stage. Fill in the `TODO` cells; reference in `solution/`. Concept: [concepts/rag/grounding-and-ocr.md](../../concepts/rag/grounding-and-ocr.md).

## Step 0: Setup

In [ ]:
from mm_eval import (SharedSpaceEmbedder, CaptionThenEmbedder, evaluate, real_vlm_hint)
# A multimodal answer can be wrong for three independent reasons: retrieval (wrong element),
# OCR-reading (misread the number), grounding (ignored the evidence). One end-to-end accuracy
# number can't tell them apart - so we grade each axis and attribute every failure to its first
# failing stage.
def show(name, r):
    print(f"{name:22} recall {r['recall']:.2f} | mean CER {r['mean_cer']:.2f} | "
          f"grounding {r['grounding_rate']:.2f} | e2e acc {r['e2e_accuracy']:.2f} | {r['attribution']}")

## Step 1: Shared-space - retrieval failures

In [ ]:
# Shared-space (CLIP-style) embedder: encodes appearance, not the text inside the image. Every
# text-in-image query fails to retrieve - and the attribution says so.
show("shared-space", evaluate(SharedSpaceEmbedder()))

## Step 2: Caption-then-embed, clean - all correct

In [ ]:
# Caption-then-embed with clean OCR and a grounded generator: everything correct.
show("caption (clean)", evaluate(CaptionThenEmbedder()))

## Step 3: Noisy OCR - grounded but wrong (failure = OCR)

In [ ]:
# TODO: run evaluate(CaptionThenEmbedder(), noisy_ids=("usr",)). Accuracy drops - but what is the
# grounding rate, and which stage does the attribution blame? Why is a misread number an OCR failure
# and not a grounding failure?
raise NotImplementedError

## Step 4: Hallucination - same accuracy, different cause (failure = grounding)

In [ ]:
# TODO: run evaluate(CaptionThenEmbedder(), ungrounded_ids=("mgn",)). It scores the SAME end-to-end
# accuracy as the noisy-OCR run. Compare the two attributions. Which bug would you fix first, and how
# would you know without the per-stage breakdown?
raise NotImplementedError

## Step 5: The real vision-language swap

In [ ]:
# Swapping in a real vision-language stack changes the stages, not the metrics:
print(real_vlm_hint())

## What you built

A multimodal RAG grader that separates the three failure sources a single accuracy number conflates. `evaluate` returns retrieval recall, mean CER, grounding rate, end-to-end accuracy, and an **attribution** that assigns every wrong answer to its first failing stage (retrieval -> OCR -> grounding). The decisive demonstration: the noisy-OCR run and the ungrounded run both score 0.75 end-to-end, but the attribution shows one is an OCR bug (grounding still 1.00 - the generator faithfully repeated a misread number) and the other is a grounding bug (CER clean - the generator ignored the evidence). Fixing the wrong one wastes a sprint.

**Where this simplifies:** the embedder, reader, and generator are deterministic stand-ins so the lab runs offline and the failures are engineered; `real_vlm_hint()` sketches the real swap (a CLIP/SigLIP embedder, a VLM/OCR reader, a VLM generator). The metrics and the attribution are the deliverable and are unchanged for a real model. The broader point: 'grounded' and 'correct' are different - an answer can be perfectly grounded in bad OCR - so a multimodal eval reports retrieval, OCR-reading, and grounding as three numbers, not one. OCR scoring itself has a trap of its own: [Lab 62](../62-ocr-reading-quality/). Concept: [concepts/rag/grounding-and-ocr.md](../../concepts/rag/grounding-and-ocr.md).